# Laya on Kaggle GPU: fill the dmroute cache

Runs Laya (English on LLMRouterBench, multilingual max_len 1024 on TwinRouterBench) over every
example and saves the call caches. Evaluation happens locally afterwards (`kaggle/README.md`).

**Settings (right panel):** Accelerator **GPU T4 x2** (not P100: current PyTorch builds dropped
Pascal), Internet **on** (Laya downloads its checkpoints from Hugging Face), and add the private
dataset **dmroute-bundle** as input. Then *Save Version → Save & Run All* so it runs without the
tab open. Outputs: `/kaggle/working/out/`.

In [ ]:
import glob, os, shutil, subprocess, sys
from pathlib import Path

src = Path(glob.glob("/kaggle/input/**/dmroute-bundle/pyproject.toml", recursive=True)[0]).parent
work = Path("/kaggle/working/bundle")  # not "dmroute": it would shadow the package
if not work.exists():
    shutil.copytree(src, work)
os.chdir(work)
print("bundle:", src)
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# Kaggle ships torch/numpy/scipy/sklearn/yaml/httpx/matplotlib; keep its CUDA torch.
!pip install -q "laya==0.3.20"
!pip install -q --no-deps -e .
# The editable install is a .pth file, read only at interpreter start; this kernel is already up.
sys.path.insert(0, str(work / "src"))
import torch, laya, transformers
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "gpus", torch.cuda.device_count())
print("laya", laya.__version__ if hasattr(laya, "__version__") else "?", "transformers", transformers.__version__)
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2"

Smoke test: both checkpoints load on CUDA, and the batched path (used for the full run) agrees
with single requests. Differences should be small (fp16 autocast on T4).

In [ ]:
os.environ.setdefault("USE_TF", "0")
import time, yaml
from dmroute.data import load
from dmroute.questions import build_questions
from dmroute.routers.base import fit_text
from laya import Router

router = Router(device="cuda")
for cfg_path, ckpt, max_len in [("configs/kaggle/llmrouterbench_laya.yaml", "english", None),
                                ("configs/kaggle/twinrouterbench_laya.yaml", "multilingual", 1024)]:
    cfg = yaml.safe_load(Path(cfg_path).read_text())
    ds = load(cfg["dataset"])
    max_chars = cfg["routers"][0]["max_chars"]
    questions = build_questions(ds.pool)
    # Same Noul->Choice conversion as LayaLocal (noul_as_choice).
    qs = {k: ({"type": "choice", "instructions": q["instructions"], "criteria": {"A": "yes", "B": "no"}}
              if q["type"] == "noul" else q) for k, q in questions.items()}
    states = [fit_text(e.text, max_chars) for e in ds.examples[:8]]
    agent = router.load(ckpt)
    t = time.perf_counter(); single = [router.predict(s, qs, model=ckpt, max_len=max_len) for s in states]
    t_single = (time.perf_counter() - t) / len(states)
    t = time.perf_counter(); batch = agent.predict_batch(states, qs, batch_size=8, max_len=max_len)
    t_batch = (time.perf_counter() - t) / len(states)
    diff = max(abs(a["answers"][k]["probabilities"][o] - b["answers"][k]["probabilities"][o])
               for a, b in zip(single, batch) for k in qs if "probabilities" in a["answers"][k]
               for o in a["answers"][k]["probabilities"])
    print(f"{ckpt}: device {agent.device}, {len(ds.examples)} examples, "
          f"{t_single:.3f} s/req single, {t_batch:.3f} s/req batched, max |dp| single vs batch {diff:.4f}")
    assert agent.device.type == "cuda"
del router, agent
torch.cuda.empty_cache()

Full runs. With two GPUs, LLMRouterBench and TwinRouterBench run in parallel, one per GPU, each
writing its own cache file. Expect roughly 10 to 40 minutes.

In [ ]:
runs = [("lrb", "configs/kaggle/llmrouterbench_laya.yaml"),
        ("trb", "configs/kaggle/twinrouterbench_laya.yaml")]
n_gpu = torch.cuda.device_count()
procs = []
t0 = time.time()
for i, (tag, cfg_path) in enumerate(runs):
    env = {**os.environ, "USE_TF": "0", "CUDA_VISIBLE_DEVICES": str(i % n_gpu)}
    log = open(f"/kaggle/working/{tag}.log", "w")
    p = subprocess.Popen(["dmroute", "run", cfg_path], env=env, stdout=log, stderr=subprocess.STDOUT)
    procs.append((tag, p, log))
    if n_gpu < 2:  # one GPU: run one after the other
        p.wait()
# Progress every minute: rows in each cache file vs expected. dmroute itself prints nothing
# until a run finishes. Stop early if Laya has fallen back to CPU (GPU OOM).
import sqlite3
expected = {"lrb": 4934, "trb": 962}

def rows(tag):
    try:
        with sqlite3.connect(f"file:.cache/laya_{tag}.sqlite?mode=ro", uri=True, timeout=5) as db:
            return db.execute("SELECT count(*) FROM calls").fetchone()[0]
    except sqlite3.Error:
        return 0

while any(p.poll() is None for _, p, _ in procs):
    time.sleep(60)
    mins = (time.time() - t0) / 60
    parts = []
    for tag, p, _ in procs:
        n = rows(tag)
        eta = (expected[tag] - n) / (n / mins) if n and p.poll() is None else 0
        state = "running" if p.poll() is None else f"exit {p.returncode}"
        parts.append(f"{tag} {n}/{expected[tag]} ({state}, ~{eta:.0f} min left)")
        if "Falling back to CPU" in Path(f"/kaggle/working/{tag}.log").read_text():
            for _, q, _ in procs:
                q.kill()
            raise RuntimeError(f"{tag}: Laya fell back to CPU (GPU OOM); lower batch_size")
    print(f"[{mins:5.1f} min] " + " | ".join(parts), flush=True)
for tag, p, log in procs:
    log.close()
    print(f"{tag}: exit {p.returncode}")
for tag, _, _ in procs:
    text = Path(f"/kaggle/working/{tag}.log").read_text()
    print(f"----- {tag} (tail) -----"); print(text[-3000:])
    # Laya moves itself to CPU for the rest of the run on a GPU OOM; those numbers are fine but
    # the run would take hours. Lower batch_size in the config and rerun if this fires.
    assert "Falling back to CPU" not in text, f"{tag}: Laya fell back to CPU (OOM)"
assert all(p.returncode == 0 for _, p, _ in procs)

In [ ]:
import sqlite3
out = Path("/kaggle/working/out")
out.mkdir(exist_ok=True)
# Distinct (state, questions) keys: 4,934 prompts; 970 TRB steps collapse to 962 after the 4k cut.
expected = {"laya_lrb.sqlite": 4934, "laya_trb.sqlite": 962}
for name, n_expected in expected.items():
    db = sqlite3.connect(f".cache/{name}")
    rows = db.execute("SELECT backend, count(*), avg(latency_s) FROM calls GROUP BY backend").fetchall()
    db.close()
    print(name, rows)
    shutil.copy2(f".cache/{name}", out / name)
    assert sum(r[1] for r in rows) == n_expected, f"{name}: incomplete"
shutil.copytree("results", out / "results", dirs_exist_ok=True)
for tag in ("lrb", "trb"):
    shutil.copy2(f"/kaggle/working/{tag}.log", out / f"{tag}.log")
os.chdir("/kaggle/working")
shutil.rmtree(work)  # keep the saved output small: only out/ and the logs remain
!ls -la /kaggle/working/out